In [0]:
from pyspark.sql import Row
from datetime import datetime

In [0]:
%sql
create or replace table aws_databricks.raw.sensor_reading(
    sensor_id string,
    reading_value int,
    recorded_at timestamp
)
using delta

In [0]:
source_data = [
    Row(
        "5001",
        300000000000000,
        datetime(2026,1,1,12,0,0)
),
    Row(
      "5002",
      500000000000000,
      datetime(2026,1,1,12,0,0)
    )
    ]

In [0]:
df = spark.createDataFrame(source_data,"sensor_id string,reading_value long,recorded_at timestamp")

      

In [0]:
df.write.mode("append").option("mergeSchema","true").saveAsTable("aws_databricks.raw.sensor_reading")

In [0]:
%sql
Alter table aws_databricks.raw.sensor_reading
set tblproperties("delta.enableTypeWidening" = "true")

In [0]:
df.write.mode("append").option("mergeSchema","true").saveAsTable("aws_databricks.raw.sensor_reading")

In [0]:
%sql
select * from aws_databricks.raw.sensor_reading

In [0]:
%sql
create table aws_databricks.raw.iot_events(
    id int,
    updated_at timestamp,
    payload variant
)
using delta

In [0]:
%sql
insert into aws_databricks.raw.iot_events values 
(1,"2026-01-01 12:00:00",
parse_json('{"sensor_id":"5001","temp":23.4,"unit":"c","battery":0.92}')),
(2,"2026-01-01 12:01:00",
parse_json('{"sensor_id":"5002","temp":25.4,"unit":"b"}')),
(3,"2026-01-01 12:02:00",
parse_json('{"sensor_id":"5003","pressure":1013.2,"unit":"a","location":{"lat":51.5,"lon":0.51}}')),
(4,"2026-01-01 12:03:00",
parse_json('{"sensor_id":"5004","temp":19.8,"unit":"c","collabaration":{"last_date":"2026-01-01","offset":-0.3}}')),
(5,"2026-01-01 12:04:00",
parse_json('{"sensor_id":"5005","temp":22.1,"unit":"c","battery":0.78,"tags":["indoor","labs"]}'))


In [0]:
%sql
select *,
payload:battery::double as battery,
payload:sensor_id::string as sensor_id,
payload:unit::string as unit,
payload:collabaration:last_date::date as last_date,
payload:location:lat::double as lat,
payload:temp::double as temp,
payload:tags::string as tags
from aws_databricks.raw.iot_events